# 🤗 x 🦾: Training ACT with LeRobot Notebook

**LeRobot ACT training notebook** 에 오신 것을 환영합니다! 이 노트북은 [🤗 LeRobot](https://github.com/huggingface/lerobot) 라이브러리를 사용하여 모방 학습(imitation learning) 정책을 학습시킬 수 있는 즉시 실행 가능한 환경을 제공합니다.

이 예제에서는 [Hugging Face Hub](https://huggingface.co/)에 호스팅된 데이터셋을 사용하여 `ACT` policy를 train시키고, 선택적으로 [Weights & Biases (wandb)](https://wandb.ai/)를 통해 학습 지표를 추적합니다.

## ⚙️ Requirements

- 학습 데이터가 포함된 Hugging Face 데이터셋 저장소 ID (`--dataset.repo_id=YOUR_USERNAME/YOUR_DATASET`)
- 선택사항: 학습 시각화를 원하는 경우 [wandb](https://wandb.ai/) 계정
- 권장사항: 빠른 학습을 위한 GPU 런타임 (예: NVIDIA A100)

## ⏱️ Expected Training Time

`ACT` policy로 100,000 스텝을 학습하는 데 NVIDIA A100 GPU 기준 **약 1.5시간이 소요** 됩니다. 성능이 낮은 GPU나 CPU에서는 훨씬 더 오래 걸릴 수 있습니다.

## Example Output

모델 체크포인트, 로그, 학습 그래프가 지정된 `--output_dir`에 저장됩니다. `wandb`를 활성화한 경우, wandb 프로젝트 대시보드에서도 진행 상황을 확인할 수 있습니다.

## Install conda

이 셀은 `condacolab`을 사용하여 Google Colab 내부에 완전한 Conda 환경을 구성합니다.

In [ ]:
!pip install -q condacolab

import condacolab

condacolab.install()

## Install LeRobot

이 셀은 Hugging Face에서 `lerobot` repository를 클론하고, FFmpeg(버전 7.1.1)를 설치한 후, 패키지를 편집 가능 모드로 설치합니다.

In [ ]:
!git clone https://github.com/huggingface/lerobot.git

!conda install ffmpeg=7.1.1 -c conda-forge

!cd lerobot && pip install -e .

## Weights & Biases login

이 셀은 실험 추적 및 로깅을 활성화하기 위해 Weights & Biases(wandb)에 로그인합니다.

In [ ]:
!wandb login

## Start training ACT with LeRobot

이 셀은 `lerobot` 라이브러리의 `train.py` 스크립트를 실행하여 robot control policy을 학습시킵니다.

다음 인수들을 본인의 설정에 맞게 조정하세요:

1. `--dataset.repo_id=YOUR_HF_USERNAME/YOUR_DATASET`: 데이터셋이 저장된 Hugging Face Hub 저장소 ID로 변경하세요. 예: `pepijn223/il_gym0`

2. `--policy.type=act`: 사용할 정책 구성을 지정합니다. `act`는 [configuration_act.py](../lerobot/common/policies/act/configuration_act.py)를 참조하며, 데이터셋 설정(예: 모터 및 카메라 수)에 자동으로 적응합니다.

3. `--output_dir=outputs/train/...`: 학습 로그와 모델 체크포인트가 저장될 디렉토리입니다.

4. `--job_name=...`: 이 학습 작업의 이름으로, 로깅 및 Weights & Biases에서 사용됩니다.

5. `--policy.device=cuda`: NVIDIA GPU에서 학습하는 경우 `cuda`를 사용하세요. Apple Silicon의 경우 `mps`, GPU가 없는 경우 `cpu`를 사용하세요.

6. `--wandb.enable=true`: 학습 진행 상황 시각화를 위해 Weights & Biases를 활성화합니다. 실행 전에 `wandb login`을 통해 로그인해야 합니다.

In [ ]:
import os

# 환경변수 설정 (Colab에서는 !export로 하면 셀 간에 유지 안되므로 이렇게 하는 게 안전)
os.environ["TASK_NAME"] = "pick_and_place"
os.environ["HF_USER"] = "HueyWoo"

In [ ]:
!cd lerobot && lerobot-train \
  --dataset.repo_id=${HF_USER}/SoArm_${TASK_NAME} \
  --policy.repo_id=${HF_USER}/ACT_SoArm_${TASK_NAME} \
  --policy.type=act \
  --policy.device=cuda \
  --job_name=act_so101 \
  --output_dir=outputs/train/act_so101/${TASK_NAME} \
  --wandb.enable=true \
  --steps=50_000 \
  --save_checkpoint=true \
  --save_freq=10_000

## Login into Hugging Face Hub

학습이 완료된 후 Hugging Face Hub에 로그인하고 마지막 체크포인트를 업로드합니다.

In [ ]:
!huggingface-cli login

In [ ]:
!huggingface-cli upload ${HF_USER}/${TASK_NAME} \
  /content/lerobot/outputs/train/act_so101/${TASK_NAME}/checkpoints/last/pretrained_model

# 데이터 병합 (Dataset Merging)

## 1) 라이브러리 설치

In [ ]:
!pip install datasets huggingface_hub --quiet

## 2) 라이브러리 임포트

In [ ]:
from datasets import load_dataset, concatenate_datasets, DatasetDict
from huggingface_hub import HfApi, Repository, upload_folder
import os

## 3) 데이터셋 로드

In [ ]:
# 환경변수에서 불러오기
HF_USER = os.environ["HF_USER"]
TASK_NAME = os.environ["TASK_NAME"]

ds1 = load_dataset(f"{HF_USER}/pick_and_place", split=None)  # split=None이면 전체 DatasetDict
ds2 = load_dataset(f"{HF_USER}/pick_and_place_2", split=None)

print(ds1)
print(ds2)

## 4) 스키마 맞추기 (필요하다면)

예: ds2 에 없는 컬럼을 ds1 에 맞춰 추가하거나 리네임

In [ ]:
# 아래는 예시이므로 실제 컬럼명 확인해서 바꿔야 함
common_cols = set(ds1["train"].column_names).intersection(set(ds2["train"].column_names))

ds1 = DatasetDict({k: ds1[k].select_columns(list(common_cols)) for k in ds1})
ds2 = DatasetDict({k: ds2[k].select_columns(list(common_cols)) for k in ds2})

## 5) 병합(concatenate) – split 기준 동일해야 함

In [ ]:
merged = DatasetDict()

for split in ds1.keys():
    if split in ds2:
        merged[split] = concatenate_datasets([ds1[split], ds2[split]])
    else:
        merged[split] = ds1[split]  # 또는 ds2만 있는 split도 처리

print(merged)

## 6) 선택적으로 shuffle 및 저장

In [ ]:
for split in merged.keys():
    merged[split] = merged[split].shuffle(seed=42)

merged.save_to_disk("merged_dataset")

## 7) HF Hub에 업로드 준비

In [ ]:
repo_name = f"{HF_USER}/merged_pick_place_dataset"
api = HfApi()

# dataset 타입으로 새 레포 생성
api.create_repo(
    repo_id=repo_name,
    repo_type="dataset",
    private=False,
    exist_ok=True
)

## 8) 업로드 – 간단히 폴더 업로드 방식

In [ ]:
upload_folder(
    folder_path="merged_dataset",
    path_in_repo="",
    repo_id=repo_name,
    repo_type="dataset",
    # token=os.environ["HF_TOKEN"]
)

# 모델 추가 학습, 재학습

Fine-tuning / Continued Training / Further Pretraining

In [ ]:
!cd lerobot && lerobot-train \
  --dataset.repo_id=${HF_USER}/SoArm_${TASK_NAME} \
  --policy.repo_id=${HF_USER}/ACT_SoArm_${TASK_NAME} \
  --policy.type=act \
  --policy.device=cuda \
  --policy.pretrained_path=${HF_USER}/grabsock \
  --job_name=act_so101 \
  --output_dir=outputs/train/act_so101/${TASK_NAME} \
  --wandb.enable=true \
  --steps=50_000 \
  --save_checkpoint=true \
  --save_freq=10_000